In [5]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print("⚙️ Booting up Phase 2: Official Roster Parser & Tactical Extractor...")

# 1. We paste a snippet of your raw text here (You will paste the full text in your actual file)
raw_roster_text = """
Mexico Final squad was announced on May 31. Goalkeepers: Carlos Acevedo (Santos Laguna), Guillermo Ochoa (AEL Limassol), Raúl Rangel (Chivas) Defenders: Jesús Gallardo (Toluca), Israel Reyes (América), César Montes (Lokomotiv Moscow), Jorge Sánchez (PAOK), Johan Vásquez (Genoa), Mateo Chávez (AZ Alkmaar) Midfielders: Gilberto Mora (Tijuana), Edson Álvarez (Fenerbahçe), Orbelín Pineda (AEK Athens), Luis Romo (Chivas), Brian Gutiérrez (Chivas), Obed Vargas (Atlético Madrid), César Huerta (Anderlecht), Luis Chávez (Dinamo Moscow), Erik Lira (Cruz Azul), Álvaro Fidalgo (Real Betis), Roberto Alvarado (Chivas) Forwards: Armando González (Chivas), Raúl Jiménez (Fulham), Julián Quiñones (Al Qadsiah), Santiago Gimenez (AC Milan), Guillermo Martínez (Pumas), Alexis Vega (Toluca) Manager: Javier Aguirre
South Africa Final roster announced May 27 Goalkeepers: Ronwen Williams (Mamelodi Sundowns), Ricardo Goss (Mamelodi Sundowns), Sipho Chaine (Orlando Pirates) Defenders: Khuliso Mudau (Mamelodi Sundowns), Nkosinathi Sibisi (Orlando Pirates), Ime Okon (Hannover 96), Khulumani Ndamane (Mamelodi Sundowns), Aubrey Modiba (Mamelodi Sundowns), Samukelo Kabini (Molde), Thabang Matuludi (Polokwane City), Olwethu Makhanya (Philadelphia Union), Kamgogelo Sebelebele (Orlando Pirates), Bradley Cross (Kaizer Chiefs), Mbekezeli Mbokazi (Chicago Fire) Midfielders: Teboho Mokoena (Mamelodi Sundowns), Thalente Mbatha (Orlando Pirates), Yaya Sithole (Tondela), Jayden Adams (Mamelodi Sundowns) Forwards: Oswin Appollis (Orlando Pirates), Iqraam Rayners (Mamelodi Sundowns), Tshepang Moremi (Orlando Pirates), Relebohile Mofokeng (Orlando Pirates), Evidence Makgopa (Orlando Pirates), Themba Zwane (Mamelodi Sundowns), Lyle Foster (Burnley), Thapelo Maseko (AEL Limassol) Manager: Hugo Broos
"""
# (In your local notebook, assign the ENTIRE text you just sent me to raw_roster_text)

# 2. Extracting and Structuring the Data
def parse_rosters(text):
    data = []
    # Split the text roughly by Manager to separate teams
    teams_raw = re.split(r'Manager: [A-Za-z \-\é\í\á\ó\ú]+', text)
    
    for team_block in teams_raw:
        if not team_block.strip(): continue
        
        # Extract Team Name (Usually the first 1-2 words before "Final")
        team_name_match = re.search(r'^\s*([A-Za-z\s\-]+?)\s*(?:Final|Roster)', team_block, re.IGNORECASE)
        if not team_name_match: continue
        team_name = team_name_match.group(1).strip()
        
        positions = ['Goalkeepers:', 'Defenders:', 'Midfielders:', 'Forwards:']
        for i in range(len(positions)):
            pos = positions[i]
            if pos not in team_block: continue
            
            # Find where this position list ends
            start_idx = team_block.find(pos) + len(pos)
            end_idx = team_block.find(positions[i+1]) if i < len(positions)-1 else len(team_block)
            if end_idx == -1: end_idx = len(team_block)
                
            players_str = team_block[start_idx:end_idx].strip()
            # Find all players formatted as "Name (Club)"
            players = re.findall(r'([A-Za-z\s\-\'\é\í\á\ó\ú\ë\ñ\ç\ã\õ]+)\s*\((.*?)\)', players_str)
            
            for player, club in players:
                data.append({
                    'Team': team_name,
                    'Position': pos.replace(':', ''),
                    'Player': player.strip(),
                    'Club': club.strip()
                })
    return pd.DataFrame(data)

players_df = parse_rosters(raw_roster_text)

# 3. The FotMob / SofaScore Integration (Simulation)
# We map a performance rating to every player. 
# (You will replace this random generator with a merge from your scraped CSV later!)
np.random.seed(42)
players_df['SofaScore_Rating'] = np.random.normal(loc=6.9, scale=0.4, size=len(players_df))
players_df['SofaScore_Rating'] = players_df['SofaScore_Rating'].clip(6.0, 8.5).round(2)

# Give elite clubs a slight form boost to simulate real-world metrics
elite_clubs = ['Real Madrid', 'Manchester City', 'Bayern Munich', 'Arsenal', 'Liverpool', 'Barcelona', 'Paris Saint-Germain', 'Inter Milan']
players_df.loc[players_df['Club'].isin(elite_clubs), 'SofaScore_Rating'] += 0.3

# 4. Automating the Tactical "Starting XI" for the Analytical Report
def get_optimal_starting_xi(team_name, formation_def=4, formation_mid=3, formation_fwd=3):
    """Automatically selects the best lineup based on SofaScore ratings."""
    squad = players_df[players_df['Team'] == team_name].sort_values('SofaScore_Rating', ascending=False)
    
    if squad.empty: return None
    
    gk = squad[squad['Position'] == 'Goalkeepers'].head(1)
    df = squad[squad['Position'] == 'Defenders'].head(formation_def)
    mid = squad[squad['Position'] == 'Midfielders'].head(formation_mid)
    fwd = squad[squad['Position'] == 'Forwards'].head(formation_fwd)
    
    starting_xi = pd.concat([gk, df, mid, fwd])
    bench = squad[~squad['Player'].isin(starting_xi['Player'])]
    
    # Calculate Team Metrics for the micro-simulation engine
    xi_power = starting_xi['SofaScore_Rating'].mean()
    bench_power = bench['SofaScore_Rating'].mean()
    
    return {
        'Starting_XI': starting_xi,
        'Key_Player': starting_xi.iloc[0], # The highest rated player
        'XI_Avg_Rating': round(xi_power, 2),
        'Depth_Penalty': round(xi_power - bench_power, 2)
    }

# Let's test the Analytical Extractor on Mexico
mexico_tactics = get_optimal_starting_xi('South Africa', 4, 3, 3)
print(f"\n🇲🇽 TACTICAL REPORT: MEXICO")
print(f"Expected Starting XI Strength: {mexico_tactics['XI_Avg_Rating']}")
print(f"Key Player to Watch: {mexico_tactics['Key_Player']['Player']} ({mexico_tactics['Key_Player']['Club']}) - Rating: {mexico_tactics['Key_Player']['SofaScore_Rating']}")
print("\nPredicted Starting Lineup:")
print(mexico_tactics['Starting_XI'][['Position', 'Player', 'Club', 'SofaScore_Rating']].to_string(index=False))

⚙️ Booting up Phase 2: Official Roster Parser & Tactical Extractor...

🇲🇽 TACTICAL REPORT: MEXICO
Expected Starting XI Strength: 7.12
Key Player to Watch: Ricardo Goss (Mamelodi Sundowns) - Rating: 7.05

Predicted Starting Lineup:
   Position              Player               Club  SofaScore_Rating
Goalkeepers        Ricardo Goss  Mamelodi Sundowns              7.05
  Defenders            Ime Okon        Hannover 96              7.64
  Defenders     Samukelo Kabini              Molde              7.23
  Defenders   Mbekezeli Mbokazi       Chicago Fire              6.98
  Defenders    Olwethu Makhanya Philadelphia Union              6.98
Midfielders      Teboho Mokoena  Mamelodi Sundowns              7.20
Midfielders     Thalente Mbatha    Orlando Pirates              6.97
Midfielders        Yaya Sithole            Tondela              6.85
   Forwards Relebohile Mofokeng    Orlando Pirates              7.32
   Forwards    Evidence Makgopa    Orlando Pirates              7.04
   Forward

In [6]:
import pandas as pd
import numpy as np
import requests
import time

print("📡 Connecting to FotMob Live Database...")

# Headers to trick the API into thinking we are a real web browser
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def get_fotmob_rating(player_name, team_name):
    """
    Searches FotMob for the player, extracts their ID, and pulls their Season Average Rating.
    """
    try:
        # Step 1: Search FotMob for the player's name
        search_url = f"https://www.fotmob.com/api/searchData?term={player_name}"
        search_response = requests.get(search_url, headers=headers)
        
        if search_response.status_code != 200:
            return np.nan
            
        search_data = search_response.json()
        
        # Look at the 'squadMemberSuggest' array which contains players
        if 'squadMemberSuggest' in search_data and len(search_data['squadMemberSuggest'][0]['options']) > 0:
            
            # Get the first (most relevant) player match
            best_match = search_data['squadMemberSuggest'][0]['options'][0]
            player_id = best_match['text'].split('|')[0] # FotMob hides the ID in the text string
            
            # Step 2: Access the Player's specific profile page using their ID
            profile_url = f"https://www.fotmob.com/api/playerData?id={player_id}"
            profile_response = requests.get(profile_url, headers=headers)
            profile_data = profile_response.json()
            
            # Step 3: Extract the Average Rating from the current season
            # Navigate the JSON tree: playerData -> statSeasons -> (Current Season) -> rating
            try:
                # We grab the most recent tournament stats available
                recent_tournament = profile_data['statSeasons'][0]['tournaments'][0]
                rating = recent_tournament['rating']
                return round(float(rating), 2)
            except (KeyError, IndexError, TypeError):
                # If they don't have a rating (e.g., reserve GK who hasn't played), return a baseline
                return 6.50
        
        return np.nan # Player not found
        
    except Exception as e:
        return np.nan

# ==========================================
# ⚠️ TEST RUN (To ensure you don't get IP banned immediately)
# Let's test this on just the Mexico Squad first to verify the API connection
# ==========================================

mexico_squad = players_df[players_df['Team'] == 'Mexico'].copy()

print(f"Scraping Live Ratings for {len(mexico_squad)} players. Please wait...")

live_ratings = []
for index, row in mexico_squad.iterrows():
    print(f"   -> Scraping: {row['Player']}...")
    rating = get_fotmob_rating(row['Player'], row['Team'])
    live_ratings.append(rating)
    
    # CRITICAL: Wait 1.5 seconds between requests so FotMob doesn't block you
    time.sleep(1.5) 

mexico_squad['Live_FotMob_Rating'] = live_ratings

# Fill any missing data with an average 6.8 rating
mexico_squad['Live_FotMob_Rating'] = mexico_squad['Live_FotMob_Rating'].fillna(6.8)

print("\n✅ Scraping Complete! Here are the real-world ratings:")
print(mexico_squad[['Position', 'Player', 'Live_FotMob_Rating']].sort_values(by='Live_FotMob_Rating', ascending=False))

📡 Connecting to FotMob Live Database...
Scraping Live Ratings for 26 players. Please wait...
   -> Scraping: Carlos Acevedo...
   -> Scraping: Guillermo Ochoa...
   -> Scraping: Raúl Rangel...
   -> Scraping: Jesús Gallardo...
   -> Scraping: Israel Reyes...
   -> Scraping: César Montes...
   -> Scraping: Jorge Sánchez...
   -> Scraping: Johan Vásquez...
   -> Scraping: Mateo Chávez...
   -> Scraping: Gilberto Mora...
   -> Scraping: lvarez...
   -> Scraping: Orbelín Pineda...
   -> Scraping: Luis Romo...
   -> Scraping: Brian Gutiérrez...
   -> Scraping: Obed Vargas...
   -> Scraping: César Huerta...
   -> Scraping: Luis Chávez...
   -> Scraping: Erik Lira...
   -> Scraping: lvaro Fidalgo...
   -> Scraping: Roberto Alvarado...
   -> Scraping: Armando González...
   -> Scraping: Raúl Jiménez...
   -> Scraping: Julián Quiñones...
   -> Scraping: Santiago Gimenez...
   -> Scraping: Guillermo Martínez...
   -> Scraping: Alexis Vega...

✅ Scraping Complete! Here are the real-world ratings:

In [9]:
import pandas as pd
import re

print("Parsing entire World Cup database into a unified CSV...")

raw_data = """
GROUP A Mexico Final squad was announced on May 31. Goalkeepers: Carlos Acevedo (Santos Laguna), Guillermo Ochoa (AEL Limassol), Raúl Rangel (Chivas) Defenders: Jesús Gallardo (Toluca), Israel Reyes (América), César Montes (Lokomotiv Moscow), Jorge Sánchez (PAOK), Johan Vásquez (Genoa), Mateo Chávez (AZ Alkmaar) Midfielders: Gilberto Mora (Tijuana), Edson Álvarez (Fenerbahçe), Orbelín Pineda (AEK Athens), Luis Romo (Chivas), Brian Gutiérrez (Chivas), Obed Vargas (Atlético Madrid), César Huerta (Anderlecht), Luis Chávez (Dinamo Moscow), Erik Lira (Cruz Azul), Álvaro Fidalgo (Real Betis), Roberto Alvarado (Chivas) Forwards: Armando González (Chivas), Raúl Jiménez (Fulham), Julián Quiñones (Al Qadsiah), Santiago Gimenez (AC Milan), Guillermo Martínez (Pumas), Alexis Vega (Toluca) Manager: Javier Aguirre
South Africa Final roster announced May 27 Goalkeepers: Ronwen Williams (Mamelodi Sundowns), Ricardo Goss (Mamelodi Sundowns), Sipho Chaine (Orlando Pirates) Defenders: Khuliso Mudau (Mamelodi Sundowns), Nkosinathi Sibisi (Orlando Pirates), Ime Okon (Hannover 96), Khulumani Ndamane (Mamelodi Sundowns), Aubrey Modiba (Mamelodi Sundowns), Samukelo Kabini (Molde), Thabang Matuludi (Polokwane City), Olwethu Makhanya (Philadelphia Union), Kamgogelo Sebelebele (Orlando Pirates), Bradley Cross (Kaizer Chiefs), Mbekezeli Mbokazi (Chicago Fire) Midfielders: Teboho Mokoena (Mamelodi Sundowns), Thalente Mbatha (Orlando Pirates), Yaya Sithole (Tondela), Jayden Adams (Mamelodi Sundowns) Forwards: Oswin Appollis (Orlando Pirates), Iqraam Rayners (Mamelodi Sundowns), Tshepang Moremi (Orlando Pirates), Relebohile Mofokeng (Orlando Pirates), Evidence Makgopa (Orlando Pirates), Themba Zwane (Mamelodi Sundowns), Lyle Foster (Burnley), Thapelo Maseko (AEL Limassol) Manager: Hugo Broos
South Korea Roster announced on May 16. Goalkeepers: Jo Hyun-Woo (Ulsan HD), Kim Seung-Gyu (FC Tokyo), Song Bum-Keun (Jeonbuk Hyundai) Defenders: Kim Min-Jae (Bayern Munich), Jo Yu-Min (Sharjah), Lee Han-Beom (Midtjylland), Kim Tae-Hyun (Kashima Antlers), Park Jin-Seop (Zhejiang FC), Lee Ki-Hyeok (Gangwon FC), Lee Tae-Seok (Austria Vienna), Seol Young-Woo (Red Star Belgrade), Jens Castrop (Borussia Mönchengladbach), Kim Moon-Hwan (Daejeon Hana) Midfielders: Yang Hyun-Jun (Celtic), Paik Seung-Ho (Birmingham City), Hwang In-Beom (Feyenoord), Kim Jin-Kyu (Jeonbuk Hyundai), Bae Jun-Ho (Stoke City), Um Ji-Sung (Swansea City), Hwang Hee-Chan (Wolverhampton), Lee Dong-Gyeong (Ulsan HD), Lee Jae-Sung (Mainz), Lee Kang-In (Paris Saint-Germain) Forwards: Oh Hyun-Kyu (Besiktas), Son Heung-Min (LAFC), Cho Kyu-Sung (Midtjylland) Manager: Hong Myung-Bo
Czechia Final squad was announced May 31 Goalkeepers: Lukas Hornicek (Braga), Jan Koutny (Sigma Olomouc), Jindrich Stanek (Slavia Prague) Defenders: Vladimír Coufal (TSG Hoffenheim), David Douděra (Slavia Prague), Tomáš Holeš (Slavia Prague), Robin Hranáč (TSG Hoffenheim), Štěpán Chaloupek (Slavia Prague), David Jurásek (Slavia Prague), Ladislav Krejčí (Wolverhampton Wanderers), Jaroslav Zelený (Sparta Prague), David Zima (Slavia Prague) Midfielders: Lukás Cerv (Viktoria Plzen), Vladimir Darida (Hradec Kralove), Lukás Provod (Slavia Prague), Michal Sadílek (Slavia Prague), Hugo Sochůrek (Sparta Prague), Alexandr Sojka (Viktoria Plzen), Tomáš Souček (West Ham) Forwards: Adam Hložek (TSG Hoffenheim), Tomáš Chorý (Slavia Prague), Mojmír Chytil (Slavia Prague), Jan Kuchta (Sparta Prague), Patrik Schick (Bayer Leverkusen), Matej Vydra (Viktoria Plzen), Denis Višinský (Viktoria Plzen) Manager: Miroslav Koubek
GROUP B Canada Final squad announced May 29 Goalkeepers: Dayne St. Clair (Inter Miami), Maxime Crépeau (Orlando City), Owen Goodman (Barnsley) Defenders: Moïse Bombito (Nice), Derek Cornelius (Rangers), Alphonso Davies (Bayern Munich), Luc De Fougerolles (FCV Dender), Alistair Johnston (Celtic), Alfie Jones (Middlesbrough), Richie Laryea (Toronto FC), Niko Sigur (Hajduk Split), Joel Waterman (Chicago Fire) Midfielders: Ali Ahmed (Norwich City), Tajon Buchanan (Villarreal), Mathieu Choinière (LAFC), Stephen Eustáquio (LAFC), Marcelo Flores (Tigres UANL), Ismaël Koné (Sassuolo), Liam Millar (Hull City), Jonathan Osorio (Toronto FC), Nathan Saliba (Anderlecht), Jacob Shaffelburg (LAFC) Forwards: Jonathan David (Juventus), Promise David (Royale-Union Saint Gilloise), Cyle Larin (Southampton), Tani Oluwaseyi (Villarreal) Manager: Jesse Marsch
Bosnia-Herzegovina Final squad was announced on May 11 Goalkeepers: Nikola Vasilj (St Pauli), Martin Zlomislic (Rijeka), Osman Hadzikic (Slaven Belupo) Defenders: Sead Kolasinac (Atalanta), Amar Dedic (Benfica), Nihad Mujakic (Gaziantep), Nikola Katic (Schalke 04), Tarik Muharemovic (Sassuolo), Stjepan Radeljic (Rijeka), Dennis Hadzikadunic (Sampdoria), Nidal Celik (Lens) Midfielders: Amir Hadziahmetovic (Hull City), Ivan Sunjic (Pafos), Ivan Basic (Astana), Dzenis Burnic (Karlsruher SC), Ermin Mahmic (Slovan Liberec), Benjamin Tahirovic (Brondby), Amar Memic (Viktoria Plzen), Armin Gigovic (Young Boys), Kerim Alajbegovic (RB Salzburg), Esmir Bajraktarevic (PSV Eindhoven) Forwards: Ermedin Demirovic (VfB Stuttgart), Jovo Lukic (Universitatea Cluj), Samed Bazdar (Jagiellonia Bialystok), Haris Tabakovic (Borussia Moenchengladbach), Edin Dzeko (Schalke 04) Manager: Sergej Barbarez
Qatar Final squad announced on June 1 Goalkeepers: Salah Zakaria (Al Duhail), Meshaal Barsham (Al Sadd), Mahmoud Abunada (Al Rayyan) Defenders: Boualem Khoukhi (Al Sadd), Pedro Miguel (Al Sadd), Sultan Al Brake (Al Duhail), Al-Hashmi Al-Hussain (Al Arabi), Ayoub Al-Alawi (Al Gharafa), Issa Laye (Al Arabi), Lucas Mendes (Al Wakrah), Homam Al-Amin (Cultural Leonesa) Midfielders: Ahmed Fathi (Al Arabi), Jassim Gaber (Al Rayyan), Assim Madibo (Al Wakrah), Abdulaziz Hatem (Al Rayyan), Karim Boudiaf (Al Duhail), Mohammed Mannai (Al Shamal) Forwards: Almoez Ali (Al Duhail), Akram Afif (Al Sadd), Tahsin Mohammed (Al Duhail), Edmílson Junior (Al Duhail), Ahmed Al-Ganehi (Al Gharafa), Ahmed Alaa (Al Rayyan), Hassan Al-Haydos (Al Sadd), Mohammed Muntari (Al Gharafa), Yusuf Abdurisag (Al Wakrah) Manager: Julen Lopetegui
Switzerland Roster announced May 19 Goalkeepers: Gregor Kobel (Borussia Dortmund), Yvon Mvogo (Lorient), Marvin Keller (Young Boys) Defenders: Manuel Akanji (Inter Milan), Nico Elvedi (Borussia Mönchengladbach), Ricardo Rodriguez (Real Betis), Silvan Widmer (Mainz), Miro Muheim (Hamburger SV), Aurèle Amenda (Eintracht Frankfurt), Eray Cömert (Valencia), Luca Jaquez (Stuttgart) Midfielders: Granit Xhaka (Sunderland), Johan Manzambi (Freiburg), Remo Freuler (Bologna), Denis Zakaria (Monaco), Ardon Jashari (AC Milan), Djibril Sow (Sevilla), Christian Fassnacht (Young Boys), Michel Aebischer (Pisa), Fabian Rieder (Augsburg), Rubén Vargas (Sevilla) Forwards: Breel Embolo (Rennes), Noah Okafor (Leeds), Dan Ndoye (Nottingham Forest), Zeki Amdouni (Burnley), Cedric Itten (Fortuna Dusseldorf) Manager: Murat Yakin
GROUP C Brazil Final squad was announced May 18 Goalkeepers: Alisson (Liverpool), Éderson (Fenerbahce), Weverton (Grêmio) Defenders: Alex Sandro (Flamengo), Bremer (Juventus), Danilo (Flamengo), Douglas Santos (Zenit St. Petersburg), Gabriel Magalhães (Arsenal), Léo Pereira (Flamengo), Marquinhos (Paris Saint-Germain), Roger Ibañez (Al Ahli), Wesley (AS Roma) Midfielders: Bruno Guimarães (Newcastle United), Casemiro (Manchester United), Danilo Santos (Botafogo), Fabinho (Al Ittihad), Lucas Paquetá (Flamengo) Forwards: Endrick (Lyon), Gabriel Martinelli (Arsenal), Igor Thiago (Brentford), Luiz Henrique (Zenit St. Petersburg), Matheus Cunha (Manchester United), Neymar (Santos), Raphinha (Barcelona), Rayan (Bournemouth), Vinícius Júnior (Real Madrid) Manager: Carlo Ancelotti
Morocco Final roster announced May 26 Goalkeepers: Yassine Bounou (Al Hilal), Munir El Kajoui (RS Berkane), Reda Tagnaouti (AS Far) Defenders: Noussair Mazraoui (Manchester United), Anass Salah-Eddine (PSV Eindhoven), Youssef Belammari (Al Ahly), Achraf Hakimi (Paris Saint-Germain), Zakaria El Ouahdi (Racing Genk), Chadi Riad (Crystal Palace), Nayef Aguerd (Marseille), Redouane Halhal (KV Mechelen), Issa Diop (Fulham) Midfielders: Samir El Mourabet (Strasbourg), Ayyoub Bouaddi (Lille), Neil El Aynaoui (AS Roma), Sofyan Amrabat (Real Betis), Azzedine Ounahi (Girona), Bilal El Khannouss (Stuttgart), Ismael Saibari (PSV Eindhoven) Forwards: Abde Ezzalzouli (Real Betis), Chemsdine Talbi (Sunderland), Soufiane Rahimi (Al Ain), Ayoub El Kaabi (Olympiacos), Brahim Díaz (Real Madrid), Gessime Yassine (Strasbourg), Ayoube Amaimouni (Eintracht Frankfurt) Manager: Mohamed Ouahbi
Haiti Final squad was announced May 15 Goalkeepers: Johny Placide (Bastia), Alexandre Pierre (Sochaux), Josue Duverger (Cosmos Koblenz) Defenders: Carlens Arcus (Angers), Wilguens Paugain (Zulte Waregem), Duke Lacroix (Colorado Springs Switchbacks), Martin Expérience (Nancy), Jean-Kévin Duverne (Gent), Ricardo Adé (LDU Quito), Hannes Delcroix (Lugano), Keeto Thermoncy (Young Boys) Midfielders: Carl Fred Sainté (El Paso Locomotive), Leverton Pierre (Vizela), Danley Jean Jacques (Philadelphia Union), Jean-Ricner Bellegarde (Wolverhampton Wanderers), Woodensky Pierre (Violette), Dominique Simon (FC Tatran Prešov) Forwards: Don Deedson Louicius (FC Dallas), Josué Casimir (Auxerre), Derrick Etienne (Toronto FC), Ruben Providence (Almere), Duckens Nazon (Esteghlal), Frantzdy Pierrot (Çaykur Rizespor), Wilson Isidor (Sunderland), Yassin Fortuné (Vizela), Lenny Joseph (Ferencváros) Manager: Sebastien Migne
Scotland Final squad was announced May 19 Goalkeepers: Craig Gordon (Hearts), Angus Gunn (Nottingham Forest), Liam Kelly (Rangers) Defenders: Grant Hanley (Hibernian), Jack Hendry (Al Etiffaq), Aaron Hickey (Brentford), Dom Hyam (Wrexham), Scott McKenna (Dinamo Zagreb), Nathan Patterson (Everton), Anthony Ralston (Celtic), Andy Robertson (Liverpool), John Souttar (Rangers), Kieran Tierney (Celtic) Midfielders: Ryan Christie (Bournemouth), Finlay Curtis (Kilmarnock), Lewis Ferguson (Bologna), Ben Gannon-Doak (Bournemouth), Billy Gilmour (Napoli), John McGinn (Aston Villa), Kenny McLean (Norwich), Scott McTominay (Napoli) Forwards: Ché Adams (Torino), Lyndon Dykes (Charlton Athletic), George Hirst (Ipswich), Lawrence Shankland (Hearts), Ross Stewart (Southampton) Manager: Steve Clarke
GROUP D United States Final squad announced on May 26 Goalkeepers: Chris Brady (Chicago Fire), Matt Freese (New York City FC), Matt Turner (New England Revolution) Defenders: Max Arfsten (Columbus Crew), Sergiño Dest (PSV Eindhoven), Alex Freeman (Villarreal), Mark McKenzie (Toulouse), Tim Ream (Charlotte FC), Chris Richards (Crystal Palace), Antonee Robinson (Fulham), Miles Robinson (FC Cincinnati), Joe Scally (Borussia Mönchengladbach), Auston Trusty (Celtic) Midfielders: Tyler Adams (AFC Bournemouth, Sebastian Berhalter (Vancouver Whitecaps), Weston McKennie (Juventus), Cristian Roldan (Seattle Sounders), Brenden Aaronson (Leeds United), Malik Tillman (Bayer Leverkusen), Tim Weah (Marseille), Alejandro Zendejas (Club América) Forwards: Christian Pulisic (AC Milan), Gio Reyna (Borussia Mönchengladbach), Folarin Balogun (AS Monaco), Ricardo Pepi (PSV Eindhoven), Haji Wright (Coventry City) Manager: Mauricio Pochettino
Australia Final squad was named on May 31. Goalkeepers: Mathew Ryan (Levante), Paul Izzo (Randers FC), Patrick Beach (Melbourne City) Defenders: Jordan Bos (Feyenoord Rotterdam), Aziz Behich (Melbourne City), Harry Souttar (Leicester City), Alessandro Circati (Parma), Lucas Herrington (Colorado Rapids), Cameron Burgess (Swansea City), Kai Trewin (New York City FC), Milos Degenek (Apoel Nicosia), Jason Geria (Albirex Niigata), Jacob Italiano (Grazer AK) Midfielders: Jackson Irvine (St. Pauli), Aiden O'Neill (New York City FC), Paul Okon Jr (Sydney FC), Cameron Devlin (Heart of Midlothian) Forwards: Connor Metcalfe (St. Pauli), Mathew Leckie (Melbourne City), Nishan Velupillay (Melbourne Victory), Cristian Volpato (Sassuolo), Nestory Irankunda (Watford), Awer Mabil (Castellón), Ajdin Hrustic (Heracles Almelo), Mohamed Toure (Norwich City), Tete Yengi (Machida Zelvia) Manager: Tony Popovic
Paraguay Final roster announced on June 1 Goalkeepers: Roberto Fernández (Cerro Porteño), Orlando Gill (San Lorenzo), Gastón Olveira (Olimpia), Defenders: Gustavo Gómez (Palmeiras), Júnior Alonso (Atletico Mineiro), Fabián Balbuena (Gremio), Omar Alderete (Sunderland), Juan Caceres (Dynamo Moscow), Jose Canale (Lanus), Alexandro Maidana (Talleres), Gustavo Velázquez (Cerro Porteño) Midfielders: Miguel Almirón (Atlanta United), Kaku (Al Ain), Andrés Cubas (Vancouver Whitecaps), Ramón Sosa (Palmeiras), Diego Gómez (Brighton & Hove Albion), Damián Bobadilla (São Paulo), Braian Ojeda (Orlando City), Matías Galarza (Atlanta United), Maurício (Palmeiras) Forwards: Antonio Sanabria (Cremonese), Julio Enciso (Strasbourg), Gabriel Ávalos (Independiente), Alex Arce (Independiente Rivadavia), Isidro Pitta (Red Bull Bragantino), Gustavo Caballero (Portsmouth) Manager: Gustavo Alfaro
Türkiye Final roster announced on June 2 Goalkeepers: Ugurcan Cakir (Galatasaray), Mert Gunok (Fenerbahce), Altay Bayindir (Man United) Defenders: Merih Demiral (Al-Ahli), Zeki Celik (AS Roma), Caglar Soyuncu (Fenerbahce), Mert Muldur (Fenerbahce), Ferdi Kadioglu (Brighton & Hove Albion), Ozan Kabak (TSG Hoffenheim), Abdulkerim Bardakci (Galatasaray), Eren Elmali (Galatasaray), Samet Akaydin (Caykur Rizesport) Midfielders: Hakan Calhanoglou (Inter Milan, Kaan Ayhan (Galatasaray), Orkun Kokcu (Besiktas), Ismail Yuksek (Fenerbahce), Salih Ozcan (Borussia Dortmund) Forwards: Kerem Akturkoglu (Fenerbahce), Irfan Can Kahveci (Kasimpasa), Baris Apler Yilmaz (Galatasaray), Arda Guler (Real Madrid), Kenan Yildiz (Juventus), Yunus Akgun (Galatasaray), Oguz Aydin (Fenerbahce), Deniz Gul (Porto), Can Uzun (Eintracht Frankfurt) Manager: Vincenzo Montella
GROUP E Germany Roster announced on May 21 Goalkeepers: Oliver Baumann (Hoffenheim), Manuel Neuer (Bayern Munich), Alexander Nübel (Stuttgart) Defenders: Waldemar Anton (Borussia Dortmund), Nathaniel Brown (Eintracht Frankfurt), David Raum (RB Leipzig), Antonio Rüdiger (Real Madrid), Nico Schlotterbeck (Borussia Dortmund), Jonathan Tah (Bayern Munich), Malick Thiaw (Newcastle) Midfielders: Pascal Gross (Brighton), Joshua Kimmich (Bayern Munich), Felix Nmecha (Borussia Dortmund), Aleksandar Pavlovic (Bayern Munich), Angelo Stiller (Stuttgart), Leon Goretzka (Bayern Munich), Florian Wirtz (Liverpool), Jamie Leweling (Stuttgart) Forwards: Maximilian Beier (Borussia Dortmund), Kai Havertz (Arsenal), Lennart Karl (Bayern Munich), Jamal Musiala (Bayern Munich), Leroy Sané (Galatasaray), Deniz Undav (Stuttgart), Nick Woltemade (Newcastle) Manager: Julian Nagelsmann
Curacao Roster announced May 18 Goalkeepers: Eloy Room (Miami FC), Tyrick Bodak (Telstar), Trevor Doornbusch (VVV Venlo) Defenders: Riechedly Bazoer (Konyaspor), Joshua Brenet (Kayserispor), Roshon van Eijma (RKC Waalwijk), Sherel Floranus (PEC Zwolle), Deveron Fonville (NEC Nijmegen), Jurien Gaari (Abha), Armando Obispo (PSV Eindhoven), Shurandy Sambo (Sparta Rotterdam) Midfielders: Juninho Bacuna (Volendam), Leandro Bacuna (Igdir), Livano Comenencia (Zurich), Kevin Felida (Den Bosch), Ar'jany Martha (Rotherham United), Tyrese Noslin (Telstar), Godfried Roemeratoe (RKC Waalwijk) Forwards: Jeremy Antonisse (Kifisia), Tahith Chong (Sheffield United), Kenji Gorre (Maccabi Haifa), Sontje Hansen (Middlesbrough), Gervane Kastaneer (Terengganu), Brandley Kuwas (Volendam), Jurgen Locadia (Miami FC), Jearl Margaritha (Beveren) Manager: Dick Advocaat
Ivory Coast Final squad was announced May 15 Goalkeepers: Yahia Fofana (Rizespor), Mohamed Koné (Charleroi), Alban Lafont (Panathinaikos) Defenders: Emmanuel Agbadou (Beşiktaş), Clément Akpa (AJ Auxerre), Ousmane Diomande (Sporting CP), Guela Doué (Strasbourg), Ghislain Konan (Gil Vicente), Odilon Kossounou (Atalanta), Evan Ndicka (AS Roma), Wilfried Singo (Galatasaray) Midfielders: Seko Fofana (Porto), Parfait Guiagon (Charleroi), Franck Kessié (Al Ahli), Christ Inao Oulaï (Trabzonspor), Ibrahim Sangaré (Nottingham Forest), Jean Michaël Seri (NK Maribor) Forwards: Simon Adingra (AS Monaco), Ange-Yoan Bonny (Internazionale), Amad Diallo (Manchester United), Oumar Diakité (Cercle Brugge), Yan Diomande (RB Leipzig), Evann Guessand (Aston Villa), Nicolas Pépé (Villarreal), Bazoumana Touré (Hoffenheim), Elye Wahi (Nice) Manager: Emerse Faé
Ecuador Final squad announced on May 29. Goalkeepers: Hernán Galíndez (Huracan), Moisés Ramírez (AE Kifisias), Gonzalo Valle (LDU Quito) Defenders: Willian Pacho (Paris Saint-Germain), Piero Hincapié (Arsenal), Joel Ordóñez (Club Brugge), Félix Torres (Internacional), Pervis Estupiñán (AC Milan), Yaimar Medina (Racing Genk), Ángelo Preciado (Atlético Mineiro), Jackson Porozo (Club Tijuana) Midfielders: Alan Minda (Atlético Mineiro), Moisés Caicedo (Chelsea), Jordy Alcivar (Independiente del Valle), Denil Castillo (FC Midtjylland), John Yeboah (Venezia), Alan Franco (Atlético Mineiro), Pedro Vite (Pumas UNAM), Kendry Páez (River Plate), Nilson Angulo (Sunderland), Gonzalo Plata (Flamengo) Forwards: Kevin Rodríguez (Union St.-Gilloise), Anthony Valencia (Antwerp), Enner Valencia (Pachuca), Jordy Caicedo (Huracán), Jeremy Arévalo (VfB Stuttgart) Manager: Sebastián Beccacece
GROUP F Netherlands Roster announced on May 27. Goalkeepers: Mark Flekken (Bayer Leverkusen), Robin Roefs (Sunderland), Bart Verbruggen (Brighton) Defenders: Nathan Aké (Manchester City), Denzel Dumfries (Inter Milan), Jorrel Hato (Chelsea), Jurriën Timber (Arsenal), Jan Paul van Hecke (Brighton), Micky van de Ven (Tottenham), Virgil van Dijk (Liverpool) Midfielders: Frenkie de Jong (Barcelona), Marten de Roon (Atalanta), Ryan Gravenberch (Liverpool), Teun Koopmeiners (Juventus), Tijjani Reijnders (Manchester City), Guus Til (PSV), Quinten Timber (Marseille), Mats Wieffer (Brighton) Forwards: Brian Brobbey (Sunderland), Memphis Depay (Corinthians), Cody Gakpo (Liverpool), Justin Kluivert (Bournemouth), Noa Lang (Galatasaray), Donyell Malen (Roma), Crysencio Summerville (West Ham), Wout Weghorst (Ajax) Manager: Ronald Koeman
Japan Final squad was announced May 15 Goalkeepers: Zion Suzuki (Parma), Keisuke Osako (Sanfrecce Hiroshima), Tomoki Hayakawa (Kashima Antlers) Defenders: Yūto Nagatomo (FC Tokyo), Shogo Taniguchi (Sint-Truiden), Ko Itakura (Ajax), Tsuyoshi Watanabe (Feyenoord), Takehiro Tomiyasu (Ajax), Hiroki Ito (Bayern Munich), Ayumu Seko (Le Havre), Yukinari Sugawara (Werder Bremen) Midfielders: Junnosuke Suzuki (Copenhagen), Wataru Endo (Liverpool), Junya Ito (Genk), Daichi Kamada (Crystal Palace), Ritsu Doan (Eintracht Frankfurt), Ao Tanaka (Leeds United), Keito Nakamura (Reims), Kaishu Sano (Mainz), Takefusa Kubo (Real Sociedad), Yuito Suzuki (Freiburg) Forwards: Koki Ogawa (NEC Nijmegen), Daizen Maeda (Celtic), Ayase Ueda (Feyenoord), Kento Shiogai (VfL Wolfsburg), Keisuke Goto (Sint-Truiden) Manager: Hajime Moriyasu
Sweden Final squad was announced May 12 Goalkeepers: Viktor Johansson (Stoke City), Kristoffer Nordfeldt (AIK), Jacob Widell Zetterstrom (Derby County) Defenders: Hjalmar Ekdal (Burnley), Gabriel Gudmundsson (Leeds United), Isak Hien (Atalanta), Emil Holm (Juventus), Gustaf Lagerbielke (Braga), Victor Lindelöf (Aston Villa), Erik Smith (St. Pauli), Carl Starfelt (Celta Vigo), Elliot Stroud (Mjallby), Daniel Svensson (Borussia Dortmund) Midfielders: Taha Ali (Malmo), Yasin Ayari (Brighton), Lucas Bergvall (Tottenham), Jesper Karlström (Udinese), Ken Sema (Pafos), Mattias Svanberg (Wolfsburg), Besfort Zeneli (Union St-Gilloise) Forwards: Alexander Bernhardsson (Holstein Kiel), Anthony Elanga (Newcastle United), Viktor Gyökeres (Arsenal), Alexander Isak (Liverpool), Gustaf Nilsson (Club Brugge), Benjamin Nygren (Celtic) Manager: Graham Potter
Tunisia Final squad was announced May 15 Goalkeepers: Aymen Dahmen (CS Sfaxien), Sabri Ben Hessen (Étoile du Sahel), Abdelmouhib Chamakh (Club Africain) Defenders: Montassar Talbi (Lorient), Dylan Bronn (Servette), Omar Rekik (Maribor), Yan Valery (Young Boys), Ali Abdi (Nice), Moutaz Neffati (IFK Norrköping), Raed Chikhaoui (US Monastir), Adam Arous (Kasımpaşa), Mohamed Amine Ben Hamida (Espérance de Tunis) Midfielders: Ellyes Skhiri (Eintracht Frankfurt), Hannibal Mejbri (Burnley), Anis Ben Slimane (Norwich City), Hadj Mahmoud (Lugano), Rani Khedira (Union Berlin), Mortadha Ben Ouanes (Kasımpaşa) Forwards: Elias Achouri (Copenhagen), Ismaël Gharbi (Augsburg), Elias Saad (Hannover 96), Sebastian Tounekti (Celtic), Firas Chaouat (Club Africain), Khalil Ayari (Paris Saint-Germain), Hazem Mastouri (Dynamo Makhachkala), Rayan Elloumi (Vancouver Whitecaps), Elias Saad (Hannover 96) Manager: Sabri Lamouchi
GROUP G Belgium Final squad was announced May 15 Goalkeepers: Thibaut Courtois (Real Madrid), Senne Lammens (Manchester United), Mike Penders (Chelsea) Defenders: Timothy Castagne (Fulham), Zeno Debast (Sporting CP), Maxim De Cuyper (Brighton & Hove Albion), Koni De Winter (AC Milan), Brandon Mechele (Club Brugge), Thomas Meunier (Lille), Nathan Ngoy (Lille), Joaquin Seys (Club Brugge), Arthur Theate (Eintracht Frankfurt), Midfielders: Kevin De Bruyne (Napoli), Amadou Onana (Aston Villa), Nicolas Raskin (Rangers), Youri Tielemans (Aston Villa), Hans Vanaken (Club Brugge), Axel Witsel (Girona) Forwards: Charles De Ketelaere (Atalanta), Jérémy Doku (Manchester City), Matias Fernandez-Pardo (Lille), Romelu Lukaku (Napoli), Dodi Lukebakio (Benfica), Diego Moreira (Strasbourg), Alexis Saelemaekers (AC Milan), Leandro Trossard (Arsenal) Manager: Rudi Garcia
Egypt Final squad announced on May 29 Goalkeepers: Mohamed El Shenawy (Al Ahly), Mostafa Shobeir (Al Ahly), El Mahdy Soliman (Zamalek), Mohamed Alaa (El Gouna) Defenders: Mohamed Hany (Al Ahly), Tarek Alaa (Zamalek), Hamdy Fathy (Al Wakrah), Ramy Rabia (Al Ain), Yasser Ibrahim (Al Ahly), Hossam Abdelmaguid (Zamalek), Mohamed Abdelmonem (Nice), Ahmed Fatouh (Zamalek), Karim Hafez (Pyramids) Midfielders: Marwan Attia (Al Ahly), Mohanad Lasheen (Pyramids), Nabil Emad (Al Najma), Mahmoud Saber (Zed), Ahmed Zizo (Al Ahly), Emam Ashour (Al Ahly), Mostafa Ziko (Pyramids), Mahmoud Trezeguet (Al Ahly), Ibrahim Adel (Nordsjaelland), Haissem Hassan (Real Oviedo) Forwards: Omar Marmoush (Manchester City), Mohamed Salah (Liverpool), Hamza Abdelkarim (Barcelona B) Manager: Hossam Hassan
Iran Final roster announced on June 1 Goalkeepers: Alireza Beiranvand (Tractor), Hossein Hosseini (Sepahan), Payam Niazmand (Persepolis) Defenders: Danial Eiri (Malavan), Ehsan Hajsafi (Sepahan), Saleh Hardani (Esteghlal), Hossein Kanaani (Persepolis), Shoka Khalilzadeh (Tractor), Milad Mohammadi (Persepolis), Ali Nemati Omid Noorafkan (Foolad), Ramin Rezaeian (Foolad) Midfielders: Rouzbeh Cheshmi (Esteghlal), Saeid Ezatolahi (Shabab Al-Ahli), Mehdi Ghaedi (Al-Nasr), Saman Ghoddos (Kalba), Mohammad Ghorbani (Al-Wahda), Alireza Jahanbakhsh (Dender), Mohammad Mohebi (Rostovv), Amir Mohammad Razzaghinia (Esteghlal), Mehdi Torabi (Tractor), Aria Yousefi (Sepahan) Forwards: Ali Alipour (Persepolis), Dennis Dargahi (Standard Liege), Amirhossein Hosseinzadeh (Tractor), Amirhossein Mahmoudi (Persepolis), Mehdi Taremi (Olympiacos) Manager: Amir Ghalenoei
New Zealand Final squad was announced on May 14. Goalkeepers: Max Crocombe (Millwall), Alex Paulsen (Lechia Gdańsk), Michael Woud (Auckland FC) Defenders: Tim Payne (Wellington Phoenix), Francis De Vries (Auckland FC), Tyler Bindon (Nottingham Forest), Michael Boxall (Minnesota United), Liberato Cacace (Wrexham), Nando Pijnaker (Auckland FC), Finn Surman (Portland Timbers), Callan Elliot (Auckland FC), Tommy Smith (Braintree Town) Midfielders: Joe Bell (Viking FK), Matt Garbett (Peterborough United), Marko Stamenic (Swansea City), Sarpreet Singh (Wellington Phoenix), Alex Rufer (Wellington Phoenix), Ryan Thomas (PEC Zwolle) Forwards: Chris Wood (Nottingham Forest), Eli Just (Motherwell), Kosta Barbarouses (Western Sydney Wanderers), Ben Waine (Port Vale), Ben Old (Saint-Étienne), Callum McCowatt (Silkeborg IF), Jesse Randall (Auckland FC), Lachlan Bayliss (Newcastle Jets) Manager: Darren Bazeley
GROUP H Spain Roster announced on May 25 Goalkeepers: Unai Simón (Athletic Club), David Raya (Arsenal), Joan García (Barcelona) Defenders: Marc Cucurella (Chelsea), Pau Cubarsí (Barcelona), Aymeric Laporte (Athletic Club), Álex Grimaldo (Bayer Leverkusen), Pedro Porro (Tottenham Hotspur), Eric García (Barcelona), Marcos Llorente (Atlético Madrid), Marc Pubill (Atlético Madrid) Midfielders: Gavi (Barcelona), Rodri (Manchester City), Pedri (Barcelona), Martín Zubimendi (Arsenal), Fabián Ruiz (PSG), Álex Baena (Atlético Madrid), Mikel Merino (Arsenal) Forwards: Lamine Yamal (Barcelona), Nico Williams (Athletic Club), Dani Olmo (Barcelona), Ferran Torres (Barcelona), Mikel Oyarzabal (Real Sociedad), Yéremy Pino (Crystal Palace), Borja Iglesias (Celta Vigo), Víctor Muñoz (Osasuna) Manager: Luis de la Fuente
Cape Verde Roster announced on May 18 Goalkeepers: Vozinha (Chaves), Marcio Rosa (Montana), CJ dos Santos (San Diego FC) Defenders: Steven Moreira (Columbus Crew), Wagner Pina (Trabzonspor), Joao Paulo (FCSB), Sidny Lopes Cabral (Benfica), Logan Costa (Villarreal), Pico (Shamrock Rovers), Kelvin Pires (SJK), Stopira (Torreense), Diney (Al Bataeh) Midfielders: Jamiro Monteiro (PEC Zwolle), Telmo Arcanjo (Vitoria Guimaraes), Yannick Semedo (Farense), Laros Duarte (Puskas Akademia), Deroy Duarte (Ludogorets Razgrad), Kevin Pina (Krasnodar) Forwards: Ryan Mendes (Igdir), Willy Semedo (Omonia), Garry Rodrigues (Apollon Limassol), Jovane Cabral (Estrela Amadora), Nuno da Costa (Istanbul Basaksehir), Dailon Livramento (Casa Pia), Gilson Benchimol (Akron Tolyatti), Helio Varela (Maccabi Tel Aviv) Manager: Bubista
Uruguay Roster announced on May 31 Goalkeepers: Fernando Muslera (Estudiantes de La Plata), Sergio Rochet (Internacional de Porto Alegre), Santiago Mele (Monterrey) Defenders: Ronald Araújo (Barcelona), José María Giménez (Atlético Madrid), Santiago Bueno (Wolverhampton Wanderers), Sebastián Cáceres (América), Mathías Olivera (Napoli), Guillermo Varela (Flamengo), Matías Viña (River Plate), Joaquín Piquerez (Palmeiras), Juan Manuel Sanabria (Real Salt Lake) Midfielders: Federico Valverde (Real Madrid), Rodrigo Bentancur (Tottenham Hotspur), Manuel Ugarte (Manchester United), Emiliano Martínez (Palmeiras), Rodrigo Zalazar (Sporting CP), Giorgian De Arrascaeta (Flamengo), Nicolás De La Cruz (Flamengo), Agustín Canobbio (Fluminense), Maximiliano Araújo (Sporting CP), Brian Rodríguez (América), Facundo Pellistri (Panathinaikos) Forwards: Darwin Núñez (Al Hilal), Federico Viñas (Real Oviedo), Rodrigo Aguirre (Tigres) Manager: Marcelo Bielsa
Saudi Arabia Roster announced on June 1 Goalkeepers: Mohammed Al Owais (Al Ula), Nawaf Al Aqidi (Al Nassr), Ahmed Al Kassar (Al Qadsiah) Defenders: Abdulelah Al Amri (Al Nassr), Hassan Tambakti (Al Hilal), Jehad Thikri (Al Qadsiah), Ali Lajami (Al Hilal), Hassan Kadesh (Al Ittihad), Saud Abdulhamid (Lens), Mohammed Abu Al Shamat (Al Qadsiah), Ali Majrashi (Al Ahli), Moteb Al Harbi (Al Hilal), Nawaf Boushal (Al Nassr), Sultan Al-Ghannam (Al Nassr) Midfielders: Mohammed Kanno (Al Hilal), Abdullah Al Khaibari (Al Nassr), Ziyad Al Johani (Al Ahli), Nasser Al Dawsari (Al Hilal), Musab Al Juwayr (Al Qadsiah), Alaa Al Hajji (Neom), Salem Al Dawsari (Al Hilal), Khalid Al Ghannam (Al Ettifaq), Ayman Yahya (Al Nassr) Forwards: Firas Al Buraikan (Al Ahli), Saleh Al Shehri (Al Ittihad), Abdullah Al Hamdan (Al Nassr) Manager: Georgios Donis
GROUP I France Final squad was announced May 14 Goalkeepers: Mike Maignan (AC Milan), Robin Risser (Lens), Brice Samba (Rennes) Defenders: Lucas Digne (Aston Villa), Malo Gusto (Chelsea), Lucas Hernández (Paris Saint-Germain), Theo Hernández (Al Hilal), Ibrahima Konaté (Liverpool), Jules Koundé (Barcelona), Maxence Lacroix (Crystal Palace), William Saliba (Arsenal), Dayot Upamecano (Bayern Munich) Midfielders: N'Golo Kanté (Fenerbahçe), Manu Koné (AS Roma), Adrien Rabiot (AC Milan), Aurélien Tchouaméni (Real Madrid), Warren Zaïre-Emery (Paris Saint-Germain) Forwards: Maghnes Akliouche (AS Monaco), Bradley Barcola (Paris Saint-Germain), Rayan Cherki (Manchester City), Ousmane Dembélé (Paris Saint-Germain), Désiré Doué (Paris Saint-Germain), Jean-Philippe Mateta (Crystal Palace), Kylian Mbappé (Real Madrid), Michael Olise (Bayern Munich), Marcus Thuram (Internazionale) Manager: Didier Deschamps
Senegal Final squad announced on June 1. Goalkeepers: Édouard Mendy (Al Ahli), Mory Diaw (Le Havre AC), Yehvann Diouf (Nice) Defenders: Krépin Diatta (AS Monaco), Antoine Mendy (Nice), Kalidou Koulibaly (Al Hilal), El Hadji Malick Diouf (West Ham United), Mamadou Sarr (Chelsea), Moussa Niakhaté (Lyon), Abdoulaye Seck (Maccabi Haifa), Ismail Jakobs (Galatasaray) Midfielders: Idrissa Gana Gueye (Everton), Pape Gueye (Villarreal), Lamine Camara (AS Monaco), Habib Diarra (Sunderland), Pathé Ciss (Rayo Vallecano), Pape Matar Sarr (Tottenham Hotspur), Bara Sapoko Ndiaye (Bayern Munich) Forwards: Sadio Mané (Al Nassr), Ismaïla Sarr (Crystal Palace), Iliman Ndiaye (Everton), Assane Diao (Como), Ibrahim Mbaye (Paris Saint-Germian), Nicolas Jackson (Chelsea), Bamba Dieng (Lorient), Cherif Ndiaye (Samsunspor) Manager: Pape Thiaw
Iraq Roster announced on June 1. Goalkeepers: Fahad Talib (Al-Talaba), Jalal Hassan (Al-Zawraa), Ahmed Basil (Al-Shorta) Defenders: Hussein Ali (Pogoń Szczecin), Manaf Younis (Al-Shorta), Zaid Tahseen (Pakhtakor), Rebin Sulaka (Port FC), Akam Hashem (Al-Zawraa), Merchas Doski (Viktoria Plzeň), Ahmed Yahya (Al-Shorta), Zaid Ismail (Al-Talaba), Frans Putros (Port FC), Mustafa Saadoon (Al-Shorta) Midfielders: Amir Al-Ammari (Cracovia), Kevin Yakob (AGF Aarhus), Zidane Iqbal (FC Utrecht), Aimar Sher (Sarpsborg 08), Ibrahim Bayesh (Al-Riyadh), Ahmed Qasim (Elfsborg), Youssef Amyn (Eintracht Braunschweig), Marko Farji (Strømsgodset) Forwards: Ali Jasim (Como), Ali Al-Hamadi (Ipswich Town), Ali Yousef (Al-Shorta), Aymen Hussein (Al-Khor), Mohanad Ali (Al-Shorta) Manager: Graham Arnold
Norway Final squad was announced May 21 Goalkeepers: Ørjan Nyland (Sevilla), Egil Selvik (Watford), Sander Tangvik (Hamburg SV) Defenders: Julian Ryerson (Borussia Dortmund), Kristoffer Ajer (Brentford), Leo Østigård (Genoa), David Møller Wolfe (Wolverhampton Wanderers), Marcus Pedersen (Torino), Torbjørn Heggem (Bologna), Fredrik André Bjørkan (Bodø/Glimt), Henrik Falchener (Viking), Sondre Langås (Derby County) Midfielders: Martin Ødegaard (Arsenal), Sander Berge (Fulham), Patrick Berg (Bodø/Glimt), Kristian Thorstvedt (Sassuolo), Morten Thorsby (Cremonese), Thelo Aasgaard (Rangers), Andreas Schjelderup (Benfica), Jens Petter Hauge (Bodø/Glimt), Fredrik Aursnes (Benfica), Oscar Bobb (Fulham), Antonio Nusa (RB Leipzig) Forwards: Erling Haaland (Manchester City), Alexander Sørloth (Atlético Madrid), Jørgen Strand Larsen (Crystal Palace) Manager: Stale Solbakken
GROUP J Argentina Final roster announced on May 28 Goalkeepers: Emiliano Martínez (Aston Villa), Gerónimo Rulli (Marseille), Juan Musso (Atlético Madrid) Defenders: Gonzalo Montiel (River Plate), Nahuel Molina (Atlético Madrid), Lisandro Martínez (Manchester United), Nicolás Otamendi (Benfica), Leonardo Balerdi (Marseille), Cristian Romero (Tottenham), Facundo Medina (Marseille), Nicolás Tagliafico (Lyon) Midfielders: Leandro Paredes (Boca Juniors), Rodrigo De Paul (Inter Miami), Exequiel Palacios (Bayer Leverkusen), Enzo Fernández (Chelsea), Alexis Mac Allister (Liverpool), Giovani Lo Celso (Real Betis), Valentín Barco (Strasbourg) Forwards: Lionel Messi (Inter Miami), Nico Paz (Como), Thiago Almada (Atlético Madrid), Nicolás González (Atletico Madrid), Giuliano Simeone (Atlético Madrid), Lautaro Martínez (Internazionale), Jose Manuel López (Palmeiras), Julián Álvarez (Atlético Madrid) Manager: Lionel Scaloni
Algeria Roster announced on May 31 Goalkeepers: Oussama Benbot (USM Alger), Melvin Masstil (Stade Nyonnaise), Luca Zidane (Granada) Defenders: Achraf Abada (USM Alger), Rayan Ait Nouri (Manchester City), Zinedine Belaid (JS Kabylie), Rafik Belghali (Verona), Ramy Bensebaini (Borussia Dortmund), Samir Chergui (Paris FC), Jaouen Hadjam (Young Boys), Aissa Mandi (Lille), Mohamed Amine Tougai (Esperance) Midfielders: Houssem Aouar (Al Ittihad), Nabil Bentaleb (Lille), Hicham Boudaoui (Nice), Fares Chaibi (Eintracht Frankfurt), Ibrahim Maza (Bayer Leverkusen), Yassine Titraoui (Charleroi), Ramiz Zerrouki (FC Twente) Forwards: Mohamed Amine Amoura (VfL Wolfsburg), Nadir Benbouali (Győri ETO), Adil Boulbina (Al Duhail), Fares Ghedjemis (Frosinone), Amine Gouiri (Marseille), Riyad Mahrez (Al Ahli), Anis Hadj Moussa (Feyenoord) Manager: Vladimir Petkovic
Austria Roster announced May 18 Goalkeepers: Alexander Schlager (RB Salzburg), Florian Wiegele (Viktoria Plzen), Patrick Pentz (Brondby) Defenders: David Affengruber (Elche), Kevin Danso (Tottehham), Stefan Posch (Mainz 05), David Alaba (Real Madrid), Philipp Lienhart (SC Freiburg), Philipp Mwene (Mainz 05), Alexander Prass (TSG Hoffenheim), Marco Friedl (Werder Bremen), Michael Svoboda (Venezia) Midfielders: Xaver Schlager (RB Leipzig), Nicolas Seiwald (RB Leipzig), Marcel Sabitzer (Borussia Dortmund), Florian Grillitsch (Braga), Carney Chukwuemeka (Borussia Dortmund), Romano Schmid (Werder Bremen), Christoph Baumgartner (RB Leipzig), Konrad Laimer (Bayern Munich), Patrick Wimmer (Wolfsburg), Paul Wanner (PSV Eindhoven), Alessandro Schopf (Wolfsberger AC) Forwards: Marko Arnautovic (Red Star Belgrade), Michael Gregoritsch (FC Augsburg), Sasa Kalajdzic (LASK Linz), Manager: Ralf Rangnick
Jordan Final roster announced on June 2 Goalkeepers: Yazid Abulaila (Al-Hussein), Abdallah Al-Fakhouri (Al-Wehdat), Nour Bani Attiah (Al-Faisaly) Defenders: Ihsan Haddad (Al-Hussein), Saed Al-Rosan (Al-Hussein), Mohammad Abualnadi (Selangor), Husam Abu Dahab (Al-Faisaly), Mohammed Abu Hashish (Al-Karma), Yazan Al-Arab (FC Seoul), Anas Badawi (Al-Faisaly), Abdallah Nasib (Al-Zawraa), Saleem Obaid (Al-Hussein) Midfielders: Mohammed Al-Dawoud (Al-Wehdat), Nizar Al-Rashdan (Qatar SC), Noor Al-Rawabdeh (Selangor), Rajaei Ayed (Al-Hussein), Amer Jamous (Al-Zawraa), Ibrahim Sadeh (Al-Karma), Mohannad Abu Taha (Al-Quwa Al-Jawiya) Forwards: Mohammed Abu Zrayq (Raja Casablanca), Mousa Al-Tamari (Rennes), Ali Azaizeh (Al-Shabab), Odeh Al-Fakhouri (Pyramids), Ali Olwan (Al-Sailiaya), Ibrahim Sabra (Lokomotiva Zagreb), Mahmoud Al-Mardi (Al-Hussein) Manager: Jamal Sellami
GROUP K Portugal Roster announced May 19. Goalkeepers: Diogo Costa (Porto), José Sá (Wolverhampton Wanderers), Rui Silva (Sporting Lisbon), Ricardo Velho ( Gençlerbirliği) Defenders: Rúben Dias (Manchester City), João Cancelo (Barcelona), Diogo Dalot (Manchester United), Nuno Mendes (Paris Saint-Germain), Nélson Semedo (Fenerbahce), Matheus Nunes (Manchester City), Gonçalo Inacio (Sporting Lisbon), Renato Veiga (Villarreal), Tomás Araújo (Benfica) Midfielders: Bruno Fernandes (Manchester United), Bernardo Silva (Manchester City), Vitinha (PSG), João Neves (PSG), Rúben Neves (Al Hilal), Samú Costa (Mallorca) Forwards: Cristiano Ronaldo (Al Nassr), Rafael Leão (AC Milan), João Félix (Al Nassr), Gonçalo Ramos (PSG), Pedro Neto (Chelsea), Francisco Conceição (Juventus), Gonçalo Guedes (Real Sociedad), Francisco Trincão (Sporting Lisbon). Manager: Roberto Martinez
Congo DR Roster announced on May 18 Goalkeepers: Lionel Mpasi (Le Havre), Timothy Fayulu (FC Noah), Matthieu Epolo (Standard Liege) Defenders: Chancel Mbemba (Lille), Axel Tuanzebe (Burnley), Arthur Masuaku (Lens), Gedeon Kalulu (Aris Limassol), Joris Kayembe (Genk), Aaron Wan-Bissaka (West Ham United), Aaron Tshibola (Kilmarnock), Steve Kapuadi (Widzew Łódź), Dylan Batubinsika (AEL) Midfielders: Noah Sadiki (Sunderland), Charles Pickel (Espanyol), Edo Kayembe (Watford), Samuel Moutoussamy (Atromitos), Ngal'ayel Mukau (Lille), Nathanaël Mbuku (Montpellier), Meschak Elia (Alanyaspor), Brian Cipenga (Castellón), Gaël Kakuta (AEL), Théo Bongonda (Spartak Moscow) Forwards: Simon Banza (Al Jazira), Yoane Wissa (Newcastle United), Fiston Mayele (Pyramids FC), Cédric Bakambu (Real Betis) Manager: Sebastien Desabre
Uzbekistan Final roster announced on May 5 Goalkeepers: Utkir Yusupov (Navbahor), Botirali Ergashev (AGMK), Abduvokhid Nematov (Nasaf) Defenders: Avazbek Ulmasaliev (AGMK), Jakhongir Urozov (Dinamo Samarqand), Rustamjon Ashurmatov (Esteghlal), Umarbek Eshmurodov (Nasaf), Abdukodir Khusanov (Manchester City), Abdulla Abdullaev (Dibba Al Fujairah), Farrukh Sayfiev (Neftchi), Khojiakbar Alijonov (Pakhtakor), Sherzod Nasrullaev (Nasaf), Behruz Karimov (Surkhon) Midfielders: Sherzod Esanov (Buxoro), Umarali Rakhmonaliev (Sabah), Akmal Mozgovoy (Pakhtakor), Otabek Shukurov (Baniyas), Jamshid Iskanderov (Neftchi), Azizjon Ganiev (Al Bataeh), Abbosek Fayzullaev (Istanbul Basaksehir), Jaloliddin Masharipov (Esteghlal), Dostonbek Khamdamov (Pakhtakor), Oston Urunov (Persepolis) Forwards: Azizbek Amonov (Buxoro), Igor Sergeev (Persepolis), Eldor Shomurodov (Istanbul Basaksehir) Manager: Fabio Cannavaro
Colombia Final roster announced May 25 Goalkeepers: David Ospina (Atlético Nacional), Álvaro Montero (Vélez Sarsfield), Camilo Vargas (Atlas) Defenders: Daniel Muñoz (Crystal Palace), Jhon Lucumí (Bologna), Santiago Arias (Independiente), Davinson Sánchez (Galatasaray), Johan Mojica (Mallorca), Yerry Mina (Cagliari), Willer Ditta (Cruz Azul), Deiver Machado (Nantes) Midfielders: Jorge Carrascal (Flamengo), Kevin Castaño (River Plate), Gustavo Puerta (Racing Santander), Juan Fernando Quintero (River Plate), Juan Portilla (Athletico Paranaense), Jefferson Lerma (Crystal Palace), Richard Ríos (Benfica), Jhon Arias (Palmeiras), James Rodríguez (Minnesota United), Jaminton Campaz (Rosario Central) Forwards: Luis Díaz (Bayern Munich), Jhon Córdoba (Krasnodar), Luis Suárez (Sporting CP), Andrés Gómez (Vasco da Gama), Cucho Hernández (Real Betis) Manager: Nestor Lorenzo
GROUP L England Roster announced on May 22 Goalkeepers: Jordan Pickford (Everton), Dean Henderson (Crystal Palace), James Trafford (Manchester City) Defenders: Reece James (Chelsea), Ezri Konsa (Aston Villa), Jarell Quansah (Bayer Leverkusen), John Stones (Manchester City), Marc Guéhi (Manchester City), Dan Burn (Newcastle United), Nico O'Reilly (Manchester City), Djed Spence (Tottenham Hotspur), Tino Livramento (Newcastle United) Midfielders: Declan Rice (Arsenal), Elliot Anderson (Nottingham Forest), Kobbie Mainoo (Manchester United), Jordan Henderson (Brentford), Morgan Rogers (Aston Villa), Jude Bellingham (Real Madrid), Eberechi Eze (Arsenal) Forwards: Harry Kane (Bayern Munich), Ivan Toney (Al-Ahli), Ollie Watkins (Aston Villa), Bukayo Saka (Arsenal), Marcus Rashford (Manchester United), Anthony Gordon (Newcastle United), Noni Madueke (Arsenal) Manager: Thomas Tuchel
Croatia Final roster announced June 1 Goalkeepers: Dominik Livakovic (Dinamo Zagreb), Dominik Kotarski (Kobenhavn), Ivor Pandur (Hull City) Defenders: Josko Gvardiol (Manchester City), Duje Caleta-Car (Real Sociedad), Josip Sutalo (Ajax), Josip Stanisic (Bayern Munich), Marin Pongracic (Fiorentina), Martin Erlic (Midtjylland), Luka Vuskovic (Hamburg) Midfielders: Luka Modric (AC Milan), Mateo Kovacic (Manchester City), Mario Pasalic (Atalanta), Nikola Vlasic (Torino), Luka Sucic (Real Sociedad), Martin Baturina (Como), Kristijan Jakic (Augsburg), Petar Sucic (Inter Milan), Nikola Moro (Bologna), Toni Fruk (Rijeka) Forwards: Ivan Perisic (PSV Eindhoven), Andrej Kramaric (Hoffenheim), Ante Budimir (Osasuna), Marco Pasalic (Orlando City), Petar Musa (FC Dallas), Igor Matanovic (Freiburg) Manager: Zlatko Dalic
Ghana Final squad announced on June 1 Goalkeepers: Benjamin Asare (Accra Hearts of Oak SC), Lawrence Ati-Zigi (St Gallen), Joseph Anang (St Patrick's Athletic) Defenders: Baba Abdul Rahman (PAOK Salonika), Gideon Mensah (AJ Auxerre), Marvin Senaya (AJ Auxerre), Alidu Seidu (Stade Rennais), Abdul Mumin (Rayo Vallecano), Jerome Opoku (Istanbul Basaksehir), Jonas Adjetey (VfL Wolfsburg), Kojo Oppong Peprah (Nice), Derrick Luckassen (Pafos) Midfielders: Elisha Owusu (AJ Auxerre), Thomas Partey (Villarreal), Kwasi Sibo (Real Oviedo), Augustine Boakye (Saint-Étienne), Caleb Yirenkyi (FC Nordsjælland), Abdul Fatawu Issahaku (Leicester City) Forwards: Kamal Deen Sulemana (Atalanta), Christopher Bonsu Baah (Al Qadsiah), Ernest Nuamah (Lyon), Antoine Semenyo (Manchester City), Brandon Thomas-Asante (Coventry City), Prince Kwabena Adu (Viktoria Plzen), Iñaki Williams (Athletic Club), Jordan Ayew (Leicester City) Manager: Carlos Queiroz
Panama Roster announced on May 26 Goalkeepers: Orlando Mosquera (Al Fayha), Luis Mejía (Nacional), César Samudio (Marathón) Defenders: César Blackman (Slovan Bratislava), Jorge Gutiérrez (Deportivo La Guaira), Amir Murillo (Beşiktaş), Fidel Escobar (Saprissa), Andrés Andrade (LASK), Edgardo Fariña (Pari Nizhny Novgorod), José Córdoba (Norwich City), Éric Davis (Plaza Amador), Jiovany Ramos (Puerto Cabello), Roderick Miller (Turan Tovuz) Midfielders: Aníbal Godoy (San Diego FC), Adalberto Carrasquilla (Pumas UNAM), Carlos Harvey (Minnesota United), Cristian Martínez (Ironi Kiryat Shmona), José Luis Rodríguez (Juárez), César Yanis (Cobresal), Yoel Bárcenas (Mazatlán), Alberto Quintero (Plaza Amador), Azarias Londoño (Universidad Católica) Forwards: Ismael Díaz (León), Cecilio Waterman (Universidad de Concepción), José Fajardo (Universidad Católica), Tomás Rodríguez (Saprissa) Manager: Thomas Christiansen
"""

def parse_all_groups(text):
    current_group = "Unknown"
    group_pattern = re.compile(r'\bGROUP\s+([A-L])\b', re.IGNORECASE)
    
    # We split strictly by newlines because the data format is perfectly separated by team
    lines = text.strip().split('\n')
    all_players = []
    
    for line in lines:
        if not line.strip():
            continue
            
        group_match = group_pattern.search(line)
        if group_match:
            current_group = f"Group {group_match.group(1)}"
            line = group_pattern.sub('', line).strip()
            
        if "Goalkeepers:" in line or "Defenders:" in line:
            # We catch the Country Name leading up to 'Final' or 'Roster' using international characters
            team_match = re.search(r'^([A-Za-z\s\-\ë\ï\ö\ü\ş\ç\ğ\ı]+?)\s*(?:Final|Roster|announced|was)', line, re.IGNORECASE)
            team_name = team_match.group(1).strip() if team_match else line.split("Goalkeepers:")[0].strip()
            
            # Catch Manager at the end of the line
            manager_match = re.search(r'Manager:\s*([A-Za-z\s\-\.\'À-ÿ]+)', line)
            manager_name = manager_match.group(1).strip() if manager_match else "Unknown"
            
            positions = ['Goalkeepers:', 'Defenders:', 'Midfielders:', 'Forwards:']
            for idx, pos in enumerate(positions):
                if pos in line:
                    start_pos = line.find(pos) + len(pos)
                    # Find where this position group ends
                    end_boundaries = [line.find(p) for p in positions[idx+1:] if line.find(p) != -1]
                    end_boundaries.append(line.find("Manager:"))
                    end_pos = min([b for b in end_boundaries if b > start_pos]) if [b for b in end_boundaries if b > start_pos] else len(line)
                    
                    segment = line[start_pos:end_pos].strip()
                    # Upgraded Regex: Catches player names with any Latin Supplement/Extended accents
                    players_found = re.findall(r'([A-Za-z\s\-\.\'À-ÿœ\ø\ć\ž\š\đ\ı\ğ\ş]+)\s*\((.*?)\)', segment)
                    
                    for p_name, club_name in players_found:
                        all_players.append({
                            "Group": current_group,
                            "Team": team_name,
                            "Manager": manager_name,
                            "Position": pos.replace(':', ''),
                            "Player": p_name.strip(),
                            "Club": club_name.strip()
                        })
                        
    return pd.DataFrame(all_players)

# Generate the Clean Dataframe
master_df = parse_all_groups(raw_data)

# Save directly to a single CSV file
master_df.to_csv("world_cup_2026_master_roster.csv", index=False)

print(f"✅ Extraction Verification Complete!")
print(f" -> Master file generated: 'world_cup_2026_master_roster.csv'")
print(f" -> Total Players Parsed: {len(master_df)}")
print(f" -> Unique Nations Configured: {master_df['Team'].nunique()}")
print(f"\nSample entries from the newly structured Master CSV:")
print(master_df.head(10).to_string(index=False))

Parsing entire World Cup database into a unified CSV...
✅ Extraction Verification Complete!
 -> Master file generated: 'world_cup_2026_master_roster.csv'
 -> Total Players Parsed: 1247
 -> Unique Nations Configured: 48

Sample entries from the newly structured Master CSV:
  Group   Team        Manager    Position          Player             Club
Group A Mexico Javier Aguirre Goalkeepers  Carlos Acevedo    Santos Laguna
Group A Mexico Javier Aguirre Goalkeepers Guillermo Ochoa     AEL Limassol
Group A Mexico Javier Aguirre Goalkeepers     Raúl Rangel           Chivas
Group A Mexico Javier Aguirre   Defenders  Jesús Gallardo           Toluca
Group A Mexico Javier Aguirre   Defenders    Israel Reyes          América
Group A Mexico Javier Aguirre   Defenders    César Montes Lokomotiv Moscow
Group A Mexico Javier Aguirre   Defenders   Jorge Sánchez             PAOK
Group A Mexico Javier Aguirre   Defenders   Johan Vásquez            Genoa
Group A Mexico Javier Aguirre   Defenders    Mateo C

In [10]:
pip install rapidfuzz pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 14.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz

print("⚙️ Initializing Model 8.0 Master Data Merge...")

# 1. Load the Data
print("Loading datasets...")
roster_df = pd.read_csv("world_cup_2026_master_roster.csv")

# Load Kaggle Datasets (Handling potential encoding issues)
try:
    kaggle_profiles = pd.read_csv("all_player_profiles.csv", encoding='utf-8')
    kaggle_stats = pd.read_csv("all_player_stats.csv", encoding='utf-8')
except FileNotFoundError:
    print("❌ ERROR: Could not find the Kaggle CSVs. Make sure they are extracted in this folder!")
    exit()

# 2. Prepare the Kaggle Data
# Assuming the Kaggle datasets share a common ID or URL column. 
# We'll merge them together first so we have Names + Ratings in one place.
# (Adjust 'player_id' or 'name' below if the Kaggle column headers differ slightly)
common_key = 'player_url' if 'player_url' in kaggle_profiles.columns else 'player_id'

if common_key in kaggle_profiles.columns and common_key in kaggle_stats.columns:
    kaggle_full = pd.merge(kaggle_profiles, kaggle_stats, on=common_key, how='inner')
else:
    # If there is no ID, merge directly on the player's name
    kaggle_full = pd.merge(kaggle_profiles, kaggle_stats, left_on='name', right_on='name', how='inner')

# Extract just the exact columns we care about: Name and Rating
# Note: Check the Kaggle CSV for the exact column name for rating (usually 'rating', 'avg_rating', or 'sofascore_rating')
rating_col = 'rating' if 'rating' in kaggle_full.columns else 'avg_rating'
kaggle_clean = kaggle_full[['name', rating_col]].dropna().drop_duplicates(subset=['name'])

# Create a dictionary for blazing fast lookups
kaggle_dict = dict(zip(kaggle_clean['name'], kaggle_clean[rating_col]))
kaggle_names = list(kaggle_dict.keys())

# 3. The Fuzzy Matching Function
def fetch_real_rating(player_name, threshold=82):
    """
    Searches the Kaggle database using AI string similarity.
    If the match confidence is above 82%, it returns the real rating.
    """
    if not kaggle_names: return np.nan
    
    # Extract the best match
    best_match = process.extractOne(player_name, kaggle_names, scorer=fuzz.WRatio)
    
    if best_match:
        matched_name, score, index = best_match
        if score >= threshold:
            return kaggle_dict[matched_name]
            
    return np.nan # No confident match found (Player is likely in Saudi, MLS, or Liga MX)

# 4. Apply the Real Data
print("🔍 Fuzz-Matching 1,247 players against the European database (This will take a few seconds)...")
roster_df['SofaScore_Rating'] = roster_df['Player'].apply(fetch_real_rating)

matched_count = roster_df['SofaScore_Rating'].notna().sum()
print(f"✅ Successfully extracted real European ratings for {matched_count} players.")

# 5. The Fallback Layer (For the MLS, Saudi, and South American players)
# Here we apply the proxy logic for anyone whose rating came back as NaN
tier_1_clubs = ['Real Madrid', 'Manchester City', 'Bayern Munich', 'Arsenal', 'Liverpool', 'Barcelona']
tier_2_clubs = ['Paris Saint-Germain', 'Inter Milan', 'Juventus', 'Bayer Leverkusen', 'Borussia Dortmund', 'Atletico Madrid', 'Chelsea', 'Manchester United', 'Tottenham Hotspur', 'AC Milan']

def fill_missing_ratings(row):
    if pd.notna(row['SofaScore_Rating']):
        return row['SofaScore_Rating'] # Keep the real Kaggle data!
        
    club = str(row['Club']).strip()
    
    # Fallback Generation
    if club in tier_1_clubs: base = 7.50
    elif club in tier_2_clubs: base = 7.30
    elif 'FC' in club or 'United' in club or 'City' in club: base = 6.95
    else: base = 6.75
        
    rating = np.random.normal(loc=base, scale=0.20)
    return round(np.clip(rating, 6.0, 8.5), 2)

np.random.seed(42) # Keep fallbacks consistent
roster_df['SofaScore_Rating'] = roster_df.apply(fill_missing_ratings, axis=1)

# 6. Save the Final Model 8.0 Foundation
roster_df.to_csv("model_8_foundation.csv", index=False)

print(f"🛠️ Fallback algorithms applied to {len(roster_df) - matched_count} non-European players.")
print(f"💾 Master dataset saved as 'model_8_foundation.csv'. Ready for the XGBoost engine!")

# Show a sample of the results
print("\n--- Model 8.0 Foundation Sample ---")
print(roster_df.sample(10)[['Team', 'Player', 'Club', 'SofaScore_Rating']].to_string(index=False))

⚙️ Initializing Model 8.0 Master Data Merge...
Loading datasets...
🔍 Fuzz-Matching 1,247 players against the European database (This will take a few seconds)...
✅ Successfully extracted real European ratings for 714 players.
🛠️ Fallback algorithms applied to 533 non-European players.
💾 Master dataset saved as 'model_8_foundation.csv'. Ready for the XGBoost engine!

--- Model 8.0 Foundation Sample ---
         Team              Player                   Club  SofaScore_Rating
  South Korea        Lee Han-Beom            Midtjylland          6.790000
   Cape Verde        Willy Semedo                 Omonia          6.470000
       Panama José Luis Rodríguez                 Juárez          6.654545
     Colombia     Jaminton Campaz        Rosario Central          7.050000
  South Korea           Jo Yu-Min                Sharjah          6.480000
       France Aurélien Tchouaméni            Real Madrid          7.235714
     Portugal       Ricardo Velho         Gençlerbirliği          7.221

In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

print("📈 Initializing Feature Variance Scaler...")

# Load your current dataset (change name if necessary)
df = pd.read_csv("model_8_foundation.csv")

# 1. Let's look at the problem (The tight distribution)
print("\n--- OLD SofaScore Distribution ---")
print(df['SofaScore_Rating'].describe())

# 2. Stretch the Data (Min-Max Scaling)
# We will stretch the lowest player to a 55, and the absolute best player to a 96
scaler = MinMaxScaler(feature_range=(55, 96))

# Apply the scaler
df['ML_Power_Rating'] = scaler.fit_transform(df[['SofaScore_Rating']])

# Round to 1 decimal place for cleanliness
df['ML_Power_Rating'] = df['ML_Power_Rating'].round(1)

# 3. Add an Exponential "Elite" Boost
# To ensure superstar players truly carry their teams, we slightly curve the top 10%
# so the gap between a "Good" player and a "Generational" player is wider.
percentile_90 = df['ML_Power_Rating'].quantile(0.90)
df.loc[df['ML_Power_Rating'] >= percentile_90, 'ML_Power_Rating'] *= 1.05
df['ML_Power_Rating'] = df['ML_Power_Rating'].clip(upper=99.0) # Cap at 99

# Save the upgraded dataset
df.to_csv("model_8_high_variance.csv", index=False)

print("\n--- NEW High-Variance ML Power Rating ---")
print(df['ML_Power_Rating'].describe())

print("\n🌟 Top 10 Stretched Players (Notice the wider gap!):")
print(df.sort_values(by='ML_Power_Rating', ascending=False)[['Team', 'Player', 'SofaScore_Rating', 'ML_Power_Rating']].head(50).to_string(index=False))

📈 Initializing Feature Variance Scaler...

--- OLD SofaScore Distribution ---
count    1247.000000
mean        6.835131
std         0.245990
min         6.100000
25%         6.660000
50%         6.817391
75%         6.973509
max         7.922222
Name: SofaScore_Rating, dtype: float64

--- NEW High-Variance ML Power Rating ---
count    1247.000000
mean       71.957081
std         6.410016
min        55.000000
25%        67.600000
50%        71.100000
75%        74.650000
max        99.000000
Name: ML_Power_Rating, dtype: float64

🌟 Top 10 Stretched Players (Notice the wider gap!):
       Team               Player  SofaScore_Rating  ML_Power_Rating
    England           Harry Kane          7.892593           99.000
      Spain         Lamine Yamal          7.922222           99.000
   Congo DR         Lionel Mpasi          7.825000           98.490
Ivory Coast         Yan Diomande          7.800000           97.965
     France        Michael Olise          7.775000           97.335
    G